# 第 11 课｜PyTorch 张量与自动求导（Autograd）

> **课程主线：**从第 09 课的 NumPy 数组、第 10 课的注意力计算，进入**有可学习参数的计算**：前向算输出、损失衡量偏差、反向传播算梯度、更新参数。

**本课完成后，你应该能够：**
1. 用 `torch.Tensor` 建立向量、矩阵，读懂 `shape`、`dtype`、`device`；
2. 解释 `requires_grad`、`grad_fn`、`.backward()`、`.grad`；
3. 说清梯度为什么累积、如何清零、为什么更新参数要用 `torch.no_grad()`；
4. 不依赖训练框架，训练只有两个参数的模型 `y = w*x + b`；
5. 解释注意力的 `Wq/Wk/Wv` 为什么能通过同一套反向传播机制学习。

**建议 2–3 小时**，先运行，再预测，再修改。每个代码单元可以按 `Shift + Enter` 执行。练习区有留白，参考答案在后半部分。**默认使用 CPU；3060 可在以后用来学习 GPU 加速。**

## 0. 环境准备

在**终端**中安装（不要直接把命令粘贴进 Python 解释器）：

```bash
python -m pip install torch jupyterlab ipykernel
```

若你想在 RTX 3060 上使用 CUDA，请以 [PyTorch 官方安装选择器](https://pytorch.org/get-started/locally/) 提供的命令为准，注意驱动和 Python 环境。**CPU 版本也能完成本课所有实验。**

建议先使用 **Kernel → Restart Kernel and Run All Cells**，确认讲义可以独立执行，不依赖旧课程运行状态。

In [5]:
import math
import torch

print("PyTorch 版本：", torch.__version__)
print("CUDA 可用：", torch.cuda.is_available())
print("CPU 上也可以进行完整的梯度与训练实验。")
torch.set_printoptions(precision=4, sci_mode=False)

PyTorch 版本： 2.9.1+cu126
CUDA 可用： True
CPU 上也可以进行完整的梯度与训练实验。


## 1. 从 NumPy 数组到 PyTorch Tensor

`Tensor` 是 PyTorch 的核心数据对象。和 NumPy 数组相似，它也有形状、数据类型和数学运算；不同的是，PyTorch Tensor 还可以放到 CPU/GPU 上，并参与自动求导。

先预测以下三个形状：`v.shape`、`W.shape`、`v @ W` 的形状。回忆第 09 课：`(3,) @ (3,2) -> (2,)`。

In [6]:
v = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)
W = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]], dtype=torch.float32)
y = v @ W
print("v:", v, "shape:", tuple(v.shape))
print("W:\n", W, "shape:", tuple(W.shape))
print("v@W:", y, "shape:", tuple(y.shape))
assert tuple(y.shape) == (2,)
torch.testing.assert_close(y, torch.tensor([4.0, 5.0]))

v: tensor([1., 2., 3.]) shape: (3,)
W:
 tensor([[1., 0.],
        [0., 1.],
        [1., 1.]]) shape: (3, 2)
v@W: tensor([4., 5.]) shape: (2,)


### `dtype`、`ndim`、`device`

| 属性 | 作用 | 本课例子 |
|---|---|---|
| `.shape` | 每一维有多长 | `(3,2)` |
| `.ndim` | 有多少个维度 | `2` |
| `.dtype` | 元素的数据类型 | `torch.float32` |
| `.device` | 数据存放的设备 | `cpu` 或 `cuda:0` |

注意：`torch.tensor([1,2,3])` 通常推断为 **整数**；`torch.tensor([1.0,2.0,3.0])` 通常推断为**浮点数**。本课求导使用浮点 Tensor。

In [7]:
integers = torch.tensor([1, 2, 3])
floats = torch.tensor([1.0, 2.0, 3.0])
print("整数 dtype:", integers.dtype)
print("浮点 dtype:", floats.dtype)
print("形状、维度、元素数、设备：", tuple(W.shape), W.ndim, W.numel(), W.device)
assert integers.dtype == torch.int64
assert torch.is_floating_point(floats)

整数 dtype: torch.int64
浮点 dtype: torch.float32
形状、维度、元素数、设备： (3, 2) 2 6 cpu


### 矩阵计算、广播仍然适用

上一课的矩阵乘法 `@` 在 PyTorch 中几乎可以照搬。`X @ W + b` 里的 `b` 会按广播规则加到每一行，后面训练线性层时非常常见。

In [8]:
X = torch.tensor([[1.0, 2.0], [3.0, 4.0]])  # (2,2)
weight = torch.tensor([[1.0, 0.0], [0.0, 2.0]])
bias = torch.tensor([10.0, 20.0])             # (2,)
Z = X @ weight + bias
print("X@weight+bias:\n", Z)
print("形状：", tuple(Z.shape))
assert tuple(Z.shape) == (2, 2)
torch.testing.assert_close(Z, torch.tensor([[11.0, 24.0], [13.0, 28.0]]))

X@weight+bias:
 tensor([[11., 24.],
        [13., 28.]])
形状： (2, 2)


## 2. 核心转折：Tensor 也能记录计算过程

先把模型简化为一个标量参数 `w`：

$
f(w)=3w^2+2w
$

对它求导：

$
f'(w)=6w+2
$

当 `w=2`，手算函数值为 **16**，导数为 **14**。请先确认，再执行。

In [10]:
w = torch.tensor(2.0, dtype=torch.float64, requires_grad=True)
f = 3 * w**2 + 2 * w
print("w:", w, "是否需要梯度:", w.requires_grad)
print("f:", f, "计算历史:", f.grad_fn)
print("调用 backward 前 w.grad:", w.grad)
f.backward()
print("调用 backward 后 w.grad:", w.grad)
assert math.isclose(w.grad.item(), 14.0)

w: tensor(2., dtype=torch.float64, requires_grad=True) 是否需要梯度: True
f: tensor(16., dtype=torch.float64, grad_fn=<AddBackward0>) 计算历史: <AddBackward0 object at 0x00000230ADFD15A0>
调用 backward 前 w.grad: None
调用 backward 后 w.grad: tensor(14., dtype=torch.float64)


### 四个名称不要混淆

- `requires_grad=True`：告诉 PyTorch 需要为这个张量相关的运算追踪梯度。
- `grad_fn`：非叶计算结果对应的反向计算入口（不是模型参数本身）。
- `.backward()`：从当前标量输出反向传播，计算梯度。
- `.grad`：在这里存放 **叶张量** `w` 对应的累积梯度。

**注意：**`.backward()` **不会自动更新参数**。它只是算出了更新时可能用到的方向和大小。梯度的意义来自数学上的链式法则。

### 计算图与链式法则

令 \(h=3w\)，\(y=h^2\)。

$
\frac{dy}{dw}=\frac{dy}{dh}\frac{dh}{dw}=2h\cdot 3=18w
$

当 `w=2`，梯度应为 **36**。再观察中间结果是不是叶张量。

In [11]:
base = torch.tensor(2.0, requires_grad=True)
h = 3 * base
y = h**2
print("base.is_leaf:", base.is_leaf)
print("h.is_leaf:", h.is_leaf, "h.grad_fn:", h.grad_fn)
y.backward()
print("dy/dw:", base.grad.item())
assert base.is_leaf and not h.is_leaf
assert math.isclose(base.grad.item(), 36.0)

base.is_leaf: True
h.is_leaf: False h.grad_fn: <MulBackward0 object at 0x00000230AFF2B6A0>
dy/dw: 36.0


### 重要限制：非标量的输出需要告诉 backward 如何汇总

当输出是一个向量时，经常先使用 `.sum()` 或 `.mean()` 形成标量损失：

$
L(x)=x_1^2+x_2^2+x_3^2
$

那么 $\nabla_x L=[2x_1,2x_2,2x_3]$。

In [12]:
a = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
vector_output = a**2
loss = vector_output.sum()
print("vector_output.shape:", tuple(vector_output.shape))
print("loss.shape:", tuple(loss.shape))
loss.backward()
print("梯度:", a.grad)
torch.testing.assert_close(a.grad, torch.tensor([2.0, 4.0, 6.0]))

vector_output.shape: (3,)
loss.shape: ()
梯度: tensor([2., 4., 6.])


**拓展：**不做 `.sum()` 也可以对非标量调用 `.backward(gradient=...)`，但这时计算的是指定向量与雅可比矩阵的乘积。本课先统一使用**标量 loss**。

## 3. 梯度为何会累积？

这个陷阱将来训练模型经常出现：PyTorch 对叶张量的 `.grad` **默认是累加，不是自动覆盖**。为了避免误解，每次重复反向传播之前，我们会**重新构建前向计算**。

In [13]:
p = torch.tensor(2.0, requires_grad=True)
(p**2).backward()
print("第一次反向传播：", p.grad.item())  # 4
(p**2).backward()                      # 新建一次前向图
print("第二次反向传播后累积：", p.grad.item())  # 8
p.grad.zero_()
(p**2).backward()
print("清零后再次反向传播：", p.grad.item())  # 4
assert p.grad.item() == 4.0

第一次反向传播： 4.0
第二次反向传播后累积： 8.0
清零后再次反向传播： 4.0


### 为什么不要对同一张已释放的图反复 backward？

一次正常的 `.backward()` 会释放不再需要的中间计算缓存。再次对**完全相同的输出对象**反向传播，通常会报错。标准训练循环的做法是**每轮重新做一次前向计算**，而不是一直保留旧图。

下面的故障代码通过 `try/except` 演示，不会中断讲义。

In [15]:
p_repeat = torch.tensor(2.0, requires_grad=True)
old_loss = p_repeat**2
old_loss.backward()
try:
    old_loss.backward()  # 故意错误：同一张计算图重复 backward
except RuntimeError as err:
    print("预期的错误：同一张图不能直接再次 backward")
    print("错误类型：", type(err).__name__)
else:
    print("注意：当前版本或此计算路径允许重用，请以官方 autograd 规则为准。")

预期的错误：同一张图不能直接再次 backward
错误类型： RuntimeError


## 4. 从梯度到参数更新：真正迈进“训练”

最简单的梯度下降：

$
w_{\mathrm{new}}=w_{\mathrm{old}}-\eta\frac{dL}{dw}
$

以 $L(w)=w^2$ 为例，`w=2`、学习率 `η=0.1`，第一次梯度为 `4`，更新后应得到 `1.6`，损失从 `4` 变成 `2.56`。

**注意：**更新参数时，不希望把更新动作本身继续加进计算图，使用 `with torch.no_grad():`。

In [16]:
param = torch.tensor(2.0, requires_grad=True)
old_loss = param**2
old_loss.backward()
print("更新前：w=", param.item(), "loss=", old_loss.item(), "grad=", param.grad.item())
with torch.no_grad():
    param -= 0.1 * param.grad
    param.grad.zero_()
new_loss = param**2
print("更新后：w=", param.item(), "loss=", new_loss.item())
print("更新后仍需梯度:", param.requires_grad, "仍是叶张量:", param.is_leaf)
assert math.isclose(param.item(), 1.6, abs_tol=1e-6)
assert new_loss.item() < old_loss.item()

更新前：w= 2.0 loss= 4.0 grad= 4.0
更新后：w= 1.600000023841858 loss= 2.56000018119812
更新后仍需梯度: True 仍是叶张量: True


### `no_grad()` 与 `detach()` 的区别

- `with torch.no_grad():` 是一个上下文，在其中计算的常规结果不追踪反向计算。可用于推理，也可用于手动更新参数。
- `.detach()` 返回一个不参与当前梯度图的新张量视图；它不是“把模型永久切换为推理”。
- 两者**不等价于清零 `.grad`**。

尽管 `no_grad` 的常规计算结果不追踪梯度，原先的参数 `requires_grad` 不会因此被永久修改。

In [17]:
param2 = torch.tensor(2.0, requires_grad=True)
tracked = param2 * 3
with torch.no_grad():
    untracked = param2 * 3
separated = tracked.detach()
print("正常结果 requires_grad:", tracked.requires_grad)
print("no_grad 中结果 requires_grad:", untracked.requires_grad)
print("detach 后结果 requires_grad:", separated.requires_grad)
print("原参数 requires_grad:", param2.requires_grad)
assert tracked.requires_grad and not untracked.requires_grad
assert not separated.requires_grad and param2.requires_grad

正常结果 requires_grad: True
no_grad 中结果 requires_grad: False
detach 后结果 requires_grad: False
原参数 requires_grad: True


## 5. 训练一个真正会学习的两参数模型

准备 41 个训练点：`x` 从 -2 到 2，正确答案由下式产生：

$
y=3x+2
$

我们**不知道斜率和截距**，只把它们当作需要学习的参数 `w`、`b`，初始都设为 0。预测方程：

$
\hat{y}=wx+b
$

损失用均方误差（MSE）：

$
L(w,b)=\frac1N\sum_{i=1}^N(\hat y_i-y_i)^2
$

**本例的数据无噪声**，因此理论上存在零损失的最佳解 `w=3,b=2`。这是教学用的最简单例子，不能代表真实大模型的训练难度。

In [18]:
x_train = torch.linspace(-2, 2, steps=41)
y_train = 3 * x_train + 2
w_train = torch.tensor(0.0, requires_grad=True)
b_train = torch.tensor(0.0, requires_grad=True)
y_pred = w_train * x_train + b_train
loss = ((y_pred - y_train)**2).mean()
print("x_train.shape:", tuple(x_train.shape))
print("y_pred.shape:", tuple(y_pred.shape))
print("初始 loss:", round(loss.item(), 6))
print("初始 loss 是标量吗：", loss.ndim == 0)

x_train.shape: (41,)
y_pred.shape: (41,)
初始 loss: 16.599998
初始 loss 是标量吗： True


### 手算第一步梯度（可检验 autograd 是否理解正确）

$
\frac{\partial L}{\partial w}=\frac{2}{N}\sum_i (wx_i+b-y_i)x_i,
\qquad
\frac{\partial L}{\partial b}=\frac{2}{N}\sum_i (wx_i+b-y_i)
$

可以先从更小的三点数据 `x=[-1,0,1]`、`y=[-1,2,5]`、`w=b=0` 计算。你应得到：

$
\frac{\partial L}{\partial w}=-4,\qquad \frac{\partial L}{\partial b}=-4
$

In [19]:
small_x = torch.tensor([-1.0, 0.0, 1.0])
small_y = torch.tensor([-1.0, 2.0, 5.0])
w0 = torch.tensor(0.0, requires_grad=True)
b0 = torch.tensor(0.0, requires_grad=True)
small_loss = (((w0 * small_x + b0) - small_y)**2).mean()
small_loss.backward()
print("三点损失:", small_loss.item())
print("dL/dw:", w0.grad.item(), "dL/db:", b0.grad.item())
torch.testing.assert_close(w0.grad, torch.tensor(-4.0))
torch.testing.assert_close(b0.grad, torch.tensor(-4.0))

三点损失: 10.0
dL/dw: -4.0 dL/db: -4.0


### 把四个步骤放到训练循环里

1. **forward**：计算预测和损失；
2. **backward**：用 `loss.backward()` 求梯度；
3. **update**：在 `torch.no_grad()` 内更新 `w`、`b`；
4. **zero grad**：清除这轮的梯度，下轮重新算。

本例中不引入网络层或优化器，确保你能看到训练到底在做什么。

In [20]:
def train_line_manual(epochs=160, lr=0.1, device="cpu"):
    x = torch.linspace(-2, 2, steps=41, device=device)
    target = 3 * x + 2
    w = torch.tensor(0.0, device=device, requires_grad=True)
    b = torch.tensor(0.0, device=device, requires_grad=True)
    history = []

    for _ in range(epochs):
        prediction = w * x + b                   # 1. forward
        loss = ((prediction - target)**2).mean()
        history.append(loss.item())
        loss.backward()                          # 2. backward
        with torch.no_grad():                    # 3. update
            w -= lr * w.grad
            b -= lr * b.grad
            w.grad.zero_()                       # 4. zero grad
            b.grad.zero_()

    with torch.no_grad():
        final_loss = ((w * x + b - target)**2).mean().item()
    return w.item(), b.item(), history, final_loss

w_fit, b_fit, losses, last_loss = train_line_manual()
print("训练初始 loss:", round(losses[0], 6))
print("第 10 次之前 loss:", round(losses[9], 6))
print("训练最终 loss:", round(last_loss, 9))
print(f"学到的参数 w={w_fit:.6f}, b={b_fit:.6f}")
assert last_loss < 1e-6
assert abs(w_fit - 3.0) < 1e-3 and abs(b_fit - 2.0) < 1e-3

训练初始 loss: 16.599998
第 10 次之前 loss: 0.106126
训练最终 loss: 0.0
学到的参数 w=3.000000, b=2.000000


### 验收：记录变化，而不只是看“成功”

如果你想更深入理解，请把学习率 `lr` 改成 `0.01`、`0.1` 或 `1.0` 分别观察。最后一个学习率在本数据集上可能导致震荡甚至发散，**这恰好是有意义的实验结果**；不要把“训练一定下降”当作规律。

建议每次只改**一个变量**，记录初始 loss、最终 loss、参数以及是否出现 NaN。

In [21]:
for learning_rate in [0.01, 0.1]:
    wa, ba, history, fin = train_line_manual(epochs=160, lr=learning_rate)
    print(f"lr={learning_rate:.2f}  初始={history[0]:.4f}  最后={fin:.8f}  w={wa:.4f}  b={ba:.4f}")

lr=0.01  初始=16.6000  最后=0.00765306  w=2.9681  b=1.9211
lr=0.10  初始=16.6000  最后=0.00000000  w=3.0000  b=2.0000


## 6. 和标准训练代码对齐：`torch.optim.SGD`

真正的 PyTorch 项目经常使用优化器，而不是自己写 `w -= lr * w.grad`。

这里训练的还是**同一个问题**。主要变化是把“清梯度”和“更新参数”交给 `optimizer`。

In [22]:
x_opt = torch.linspace(-2, 2, steps=41)
target_opt = 3 * x_opt + 2
w_opt = torch.nn.Parameter(torch.tensor(0.0))
b_opt = torch.nn.Parameter(torch.tensor(0.0))
optimizer = torch.optim.SGD([w_opt, b_opt], lr=0.1)

for _ in range(160):
    optimizer.zero_grad()                    # 优化器清梯度
    y_hat = w_opt * x_opt + b_opt
    loss_opt = ((y_hat - target_opt)**2).mean()
    loss_opt.backward()                      # 梯度还是 autograd 算的
    optimizer.step()                         # 优化器更新参数

print("手写结果:", round(w_fit, 6), round(b_fit, 6))
print("SGD 结果:", round(w_opt.item(), 6), round(b_opt.item(), 6))
assert abs(w_opt.item() - w_fit) < 1e-4
assert abs(b_opt.item() - b_fit) < 1e-4

手写结果: 3.0 2.0
SGD 结果: 3.0 2.0


### 参数不是“凭空变聪明”的

在这个示例里：

- **数据**提供 `x` 和 `y`；
- **模型形式**规定使用 `w*x+b`；
- **损失**规定什么叫“错”；
- **autograd**计算哪里需要怎样调整；
- **优化器或手动更新**实际修改参数。

大型 Transformer 的训练环节也遵循这条主线，只是网络、数据、损失、精度和优化过程复杂得多。

## 7. 与 Transformer 的连接：Q、K、V 的矩阵也是参数

你在第 10 课直接指定了 `Q/K/V`。真实 Transformer 的某个注意力头通常先计算：

$
Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V
$

其中 `X` 是当前层输入表示，`W_Q/W_K/W_V` 是**训练得到的权重矩阵**。现在用一个有三条 token 向量的极小示例验证：**反向传播能顺着 softmax、矩阵乘法回到这些权重。**

In [23]:
X_attn = torch.tensor([[1., 2., 0.], [0., 1., 1.], [1., 1., 1.]])
Wq = torch.tensor([[0.1,0.4],[0.3,-0.2],[0.5,0.2]], requires_grad=True)
Wk = torch.tensor([[0.2,0.1],[-0.3,0.5],[0.6,0.2]], requires_grad=True)
Wv = torch.tensor([[0.4,-0.2],[0.1,0.3],[0.2,0.6]], requires_grad=True)
Q = X_attn @ Wq
K = X_attn @ Wk
V = X_attn @ Wv
scores = (Q @ K.T) / math.sqrt(Q.shape[-1])
allow = torch.tril(torch.ones((3,3), dtype=torch.bool))
attn_weights = torch.softmax(scores.masked_fill(~allow, -torch.inf), dim=-1)
attn_output = attn_weights @ V
attn_target = torch.tensor([[0.2,0.5],[0.4,0.1],[0.6,0.7]])
attn_loss = ((attn_output - attn_target)**2).mean()
attn_loss.backward()

print("X/Q/K/V/scores/output shapes:", *(tuple(t.shape) for t in [X_attn,Q,K,V,scores,attn_output]))
print("三个权重梯度形状:", *(tuple(t.grad.shape) for t in [Wq,Wk,Wv]))
print("三个权重梯度范数:", *(round(t.grad.norm().item(),6) for t in [Wq,Wk,Wv]))
assert tuple(scores.shape) == (3,3)
assert all(t.grad is not None and torch.isfinite(t.grad).all() for t in [Wq,Wk,Wv])
assert all(t.grad.norm().item() > 1e-8 for t in [Wq,Wk,Wv])

X/Q/K/V/scores/output shapes: (3, 3) (3, 2) (3, 2) (3, 2) (3, 3) (3, 2)
三个权重梯度形状: (3, 2) (3, 2) (3, 2)
三个权重梯度范数: 0.018942 0.023785 0.371649


### 到底说明了什么？

注意这个实验的证据是：`Wq.grad`、`Wk.grad`、`Wv.grad` 都具有正确形状和非零值。

**这说明梯度有路径传递到这些参数，不等于它们已经训练得很好。** 想让权重学会有用的信息交互，还需要大量数据、恰当的损失和实际的优化步骤。

## 8. 可选：查看 RTX 3060 是否能被当前 PyTorch 调用

显卡装在电脑里，不等于你安装的 PyTorch 就有 CUDA 支持。本课**无需 CUDA**；下面只有在检测到 CUDA 可用时才运行 GPU 版本的训练。`torch` 张量不能跨设备直接做矩阵运算，输入和参数应在同一设备上。

In [24]:
if torch.cuda.is_available():
    print("CUDA 设备：", torch.cuda.get_device_name(0))
    wg, bg, _, lg = train_line_manual(device="cuda")
    print(f"GPU 训练完成：w={wg:.4f}, b={bg:.4f}, loss={lg:.8f}")
else:
    print("当前 PyTorch 不支持 CUDA 或没有可用 CUDA 显卡；CPU 训练已完成，无需额外操作。")

CUDA 设备： NVIDIA GeForce RTX 3060
GPU 训练完成：w=3.0000, b=2.0000, loss=0.00000000


## 9. 常见错误排查

| 现象 | 常见原因 | 排查顺序 |
|---|---|---|
| `ModuleNotFoundError: No module named 'torch'` | 当前 Notebook 内核与安装包的 Python 环境不同 | 先查看 `import sys; print(sys.executable)`，再用该解释器安装 |
| `only Tensors of floating point ... can require gradients` | 对整数张量设置了 `requires_grad=True` | 改为 `dtype=torch.float32` 或 `float64` |
| 同一个 `loss.backward()` 第二次失败 | 上一次反向已释放计算图所需缓存 | 每轮重新计算 forward/loss |
| 梯度越来越大或叠加 | 未清零 `.grad` | 每轮 `zero_grad()`，或者明确有意做梯度累积 |
| 更新参数时出现 in-place 报错 | 在追踪梯度时直接改动叶参数 | 使用 `torch.no_grad()`，或标准优化器 |
| `RuntimeError: ... same device` | 张量有的在 CPU，有的在 GPU | 检查每个 Tensor 的 `.device` |
| loss 变大甚至 NaN | 学习率过大、数值不稳定等 | 减小学习率，观察梯度和损失、做数值检查 |

## 10. 独立练习（先做，再看参考答案）

这三道练习不依赖其他课程文件；运行完整 Notebook 不会自动替你完成它们。

### 练习 A｜独立算出多项式梯度

实现 `grad_polynomial_ex(x)`，求：

\[
g(w)=2w^2+3w+1
\]

当 `x=3`，梯度应为 **15**。要求：使用 `requires_grad=True` 和 `.backward()`，**不要直接返回手算结果**。

In [ ]:
def grad_polynomial_ex(x):
    # TODO: 将 x 转成 float64 标量 Tensor，并开启梯度
    # TODO: 构造 g=2*w**2+3*w+1，调用 backward，再返回梯度浮点数
    raise NotImplementedError("请先完成练习 A")

print("练习 A 已就绪。补全函数后自行运行：grad_polynomial_ex(3.0)，期望 15.0")

### 练习 B｜亲自更新一次参数

实现 `one_step_ex(w0, lr)`：

\[
L(w)=(w-4)^2
\]

初始 `w=0`、学习率 `0.1` 时，梯度为 `-8`，更新后应得到 `w=0.8`。要求使用 `torch.no_grad()` 更新。

In [ ]:
def one_step_ex(w0, lr):
    # TODO: 创建需要梯度的 w
    # TODO: 前向、反向、在 no_grad 中更新
    # TODO: 返回更新后的 w.item()
    raise NotImplementedError("请先完成练习 B")

print("练习 B 已就绪。补全后运行：one_step_ex(0, 0.1)，期望 0.8")

### 练习 C｜把训练目标改成另一条直线

自己实现 `train_other_line_ex(epochs=160, lr=0.1)`，把训练目标换成：

\[
y=-2x+1
\]

要求：从 `w=b=0` 开始，使用手写梯度更新。最后返回 `(w, b, final_loss)`。测试目标：**`w≈-2`、`b≈1`、`final_loss < 1e-5`**。

提示：可以模仿第 5 节的训练循环，但**不要直接把结果设为 -2 和 1**。

In [ ]:
def train_other_line_ex(epochs=160, lr=0.1):
    # TODO: 构造 x 和目标 y=-2*x+1
    # TODO: 初始化 w、b；反复前向、反向、更新和清零
    # TODO: 返回三个 Python 浮点数 (w, b, final_loss)
    raise NotImplementedError("请先完成练习 C")

print("练习 C 已就绪。完成后测试：w≈-2、b≈1、loss<1e-5")

### 额外思考｜不看运行结果，预测形状

若 `X.shape=(5,3)`、`W.shape=(3,2)`、`b.shape=(2,)`，那么：

- `X @ W + b` 形状是多少？
- 如果从这些输出构造一个**标量 loss**，`W.grad.shape` 应是多少？

提示：反向传播不会改变参数张量本身的形状。

---
## 11. 参考答案（建议独立完成后再展开阅读）

下面提供与学生练习分开的答案函数，不会覆盖你写的 `*_ex`。请尝试先解释每一步，再运行自动检查。

In [25]:
def grad_polynomial_answer(x):
    w = torch.tensor(float(x), dtype=torch.float64, requires_grad=True)
    g = 2 * w**2 + 3 * w + 1
    g.backward()
    return w.grad.item()

print("练习 A 参考答案：", grad_polynomial_answer(3.0))
assert math.isclose(grad_polynomial_answer(3.0), 15.0)

练习 A 参考答案： 15.0


In [26]:
def one_step_answer(w0, lr):
    w = torch.tensor(float(w0), requires_grad=True)
    loss = (w - 4)**2
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
    return w.item()

print("练习 B 参考答案：", one_step_answer(0, 0.1))
assert math.isclose(one_step_answer(0, 0.1), 0.8, abs_tol=1e-6)

练习 B 参考答案： 0.800000011920929


In [27]:
def train_other_line_answer(epochs=160, lr=0.1):
    x = torch.linspace(-2, 2, steps=41)
    target = -2 * x + 1
    w = torch.tensor(0.0, requires_grad=True)
    b = torch.tensor(0.0, requires_grad=True)
    for _ in range(epochs):
        loss = ((w * x + b - target)**2).mean()
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
            w.grad.zero_()
            b.grad.zero_()
    with torch.no_grad():
        final_loss = ((w * x + b - target)**2).mean().item()
    return w.item(), b.item(), final_loss

answer_w, answer_b, answer_loss = train_other_line_answer()
print(f"练习 C 参考答案：w={answer_w:.6f}, b={answer_b:.6f}, loss={answer_loss:.9f}")
assert abs(answer_w + 2) < 1e-3 and abs(answer_b - 1) < 1e-3
assert answer_loss < 1e-5

练习 C 参考答案：w=-2.000000, b=1.000000, loss=0.000000000


In [28]:
X_shape = (5, 3)
W_shape = (3, 2)
b_shape = (2,)
print("形状答案：X@W+b -> (5, 2)，W.grad.shape -> (3, 2)")
X_demo = torch.ones(X_shape)
W_demo = torch.ones(W_shape, requires_grad=True)
b_demo = torch.zeros(b_shape, requires_grad=True)
output_demo = X_demo @ W_demo + b_demo
output_demo.square().mean().backward()
print("实际输出形状：", tuple(output_demo.shape))
print("实际权重梯度形状：", tuple(W_demo.grad.shape))
assert tuple(output_demo.shape) == (5, 2)
assert tuple(W_demo.grad.shape) == (3, 2)

形状答案：X@W+b -> (5, 2)，W.grad.shape -> (3, 2)
实际输出形状： (5, 2)
实际权重梯度形状： (3, 2)


## 12. 本课验收

请用自己的话回答：

1. **`.backward()` 和 `optimizer.step()` 负责的工作有什么区别？**
2. **为什么连续执行 `backward` 可能让 `.grad` 翻倍？**
3. **为什么不应该在默认的梯度追踪状态里直接原地更新叶参数？**
4. **为什么 `Wq`、`Wk`、`Wv` 可以被训练？注意力权重和投影矩阵参数有什么区别？**
5. **`y=3x+2` 训练成功，究竟证明了什么？它不能证明什么？**

再运行配套脚本执行固定检查：

```bash
python lesson_11_pytorch_tensors_and_autograd.py
```

当环境具备 CUDA 支持、你希望比较 CPU 与 GPU 时，可以选做：

```bash
python lesson_11_pytorch_tensors_and_autograd.py --device auto
```

**下一课｜第 12 课：`torch.nn.Module`、线性层与优化器。**把本课手写的两个参数整理成可复用的神经网络，随后加入激活函数，训练一个小型多层感知机（MLP）。

**参考资料：**[PyTorch Autograd 教程](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html) · [PyTorch `no_grad` 文档](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html) · [PyTorch 安装说明](https://pytorch.org/get-started/locally/)